# Multiplex PageRank on European airlines

A multiplex network has the same nodes in several layers: here, 450 European airports, with one layer of routes for each of 37 airlines. Stored as one Tensor `A[airline, origin, destination]`, it can be ranked as a whole instead of one airline at a time.

The ranking is a multilayer PageRank, after De Domenico et al., "Ranking in interconnected multilayer networks reveals versatile nodes", *Nature Communications* 6, 6868 (2015). A random walker is at an airport *on an airline*. At each step it follows one of that airline's routes from the airport, or switches to another airline that serves the same airport (each switch weighs `omega`, each route 1); with probability 1 - `d` it jumps to a random (airline, airport) pair instead. An airport's **versatility** is the sum, over the airlines serving it, of the PageRank of its (airline, airport) pairs.

**The Tensor is experimental** (see the "Tensor Quickstart" notebook and the user guide's "Tensors" page). It needs the `suitesparse` backend with SuiteSparse:GraphBLAS 10 or later.

## The data

The EU airline multiplex of A. Cardillo, J. Gómez-Gardeñes, M. Zanin, M. Romance, D. Papo, F. del Pozo and S. Boccaletti, "Emergence of network features from multiplexity", *Scientific Reports* 3, 1344 (2013), https://doi.org/10.1038/srep01344, as distributed by the Netzschleuder network catalogue (https://networks.skewed.de/net/eu_airlines). Neither Netzschleuder nor the original page states a license; it is published as research data, so cite the paper above if you use it. The notebook downloads it (24 KB) from Netzschleuder once, checks the SHA-256 of the files it reads, and keeps it in your cache directory; nothing is redistributed here.

In [1]:
import ast
import csv
import hashlib
import io
import os
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.stats import spearmanr

import graphblas as gb
from graphblas import Tensor, binary, einsum, unary

In [2]:
URL = "https://networks.skewed.de/net/eu_airlines/files/eu_airlines.csv.zip"
SHA256 = {  # of the files read below, not of the zip, which the server may rebuild
    "edges.csv": "23ba41a3d5aba70f8714de28131462e268b1ca930447a0aaff4c3f1cde8c2a08",
    "nodes.csv": "7f088e12846d97a6d35682c4d5eb3d59fddb295247285bf2fda0e16c63f0d730",
    "gprops.csv": "74407bf1f6cf0f6583225fb33ab6b23134a395bf5080a3621f926ec5b74d16f4",
}
cache = Path(os.environ.get("XDG_CACHE_HOME", Path.home() / ".cache")) / "python-graphblas"
path = cache / "eu_airlines.csv.zip"


def read_csvs(data):
    """The CSV files in the zip ``data``, or None if one is missing or has another SHA-256."""
    try:
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            files = {name: z.read(name) for name in SHA256}
    except (zipfile.BadZipFile, KeyError):
        return None
    if any(hashlib.sha256(files[name]).hexdigest() != sha for name, sha in SHA256.items()):
        return None
    return {name: raw.decode() for name, raw in files.items()}


files = read_csvs(path.read_bytes()) if path.exists() else None
if files is None:
    data = urllib.request.urlopen(URL, timeout=60).read()
    files = read_csvs(data)
    if files is None:
        raise RuntimeError(f"{URL} does not hold the expected files (by SHA-256)")
    cache.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
sorted(files)

['edges.csv', 'gprops.csv', 'nodes.csv']

In [3]:
edges = np.loadtxt(io.StringIO(files["edges.csv"]), delimiter=",", comments="#", dtype=np.int64)
nodes = csv.reader(io.StringIO(files["nodes.csv"]))
airports = [row[1] for row in nodes if not row[0].startswith("#")]  # ICAO codes
props = {row[0]: row[1] for row in csv.reader(io.StringIO(files["gprops.csv"]))}
airlines = [name for _, name in sorted(ast.literal_eval(props["layer_key"]).items())]
src, dst, airline = edges[:, 0], edges[:, 1], edges[:, 3] - 1  # airlines are numbered from 1
n, L = len(airports), len(airlines)
n, L, len(edges)

(450, 37, 3588)

Each row of `edges.csv` is an undirected route of one airline. The Tensor holds both directions, with weight 1:

In [4]:
A = Tensor.from_coo(
    [np.concatenate([airline, airline]), np.concatenate([src, dst]), np.concatenate([dst, src])],
    1.0,
    shape=(L, n, n),
    name="A",
)
A.shape, A.nvals, A.layout

((37, 450, 450), 7176, Layout(rows=(0, 1), cols=(2,)))

Reductions over the airport axes describe each airline; reductions over the airline axis describe each airport. `degree[airline, airport]` is the number of routes the airline flies from the airport.

In [5]:
degree = A.sum(axis=2).new()
routes = degree.sum(axis=1).new().to_dense(fill_value=0).astype(int) // 2
served = degree.reduce(gb.agg.count, axis=1).new().to_dense(fill_value=0)
for i in np.argsort(-routes)[:5]:
    print(f"{airlines[i]:24s} {routes[i]:4d} routes, {served[i]:3d} airports")

Ryanair                   601 routes, 128 airports
Easyjet                   307 routes,  99 airports
Lufthansa                 244 routes, 106 airports
Air_Berlin                184 routes,  75 airports
Netjets                   180 routes,  94 airports


In [6]:
n_routes = degree.sum(axis=0).new().to_dense(fill_value=0)  # routes from each airport
n_airlines = degree.reduce(gb.agg.count, axis=0).new().to_dense(fill_value=0)
f"{degree.nvals} (airline, airport) pairs; {int((n_routes == 0).sum())} airports have no route"

'2034 (airline, airport) pairs; 33 airports have no route'

## Multiplex PageRank with the Tensor

The walk's states are the `degree.nvals` (airline, airport) pairs where the airline serves the airport; `degree.S` is their mask. With `r` the PageRank of each pair and `s` its out-weight (its routes, plus `omega` for each other airline at the airport), one step spreads `x = r / s`:

- along a route of the same airline: `einsum("lu,luv->lv", x, A)`, one contraction over the origin airport for all 37 airlines at once;
- to another airline at the same airport: `omega * (x.sum(axis=0) - x)`, broadcast back over the airlines and kept on the mask.

Every pair has at least one route, so none is dangling.

In [7]:
def multiplex_pagerank(A, omega=1.0, d=0.85, tol=1e-12, itermax=1000):
    L, n, _ = A.shape
    degree = A.sum(axis=2).new()
    N = degree.nvals
    k = degree.reduce(gb.agg.count, axis=0).new()  # airlines at each airport
    s = Tensor(gb.dtypes.FP64, (L, n))
    s(degree.S) << degree + (k - 1) * omega
    inv_s = s.apply(unary.minv).new()
    teleport = degree.apply(binary.second, (1 - d) / N).new()
    r = degree.apply(binary.second, 1 / N).new()
    for it in range(1, itermax + 1):
        x = (r * inv_s).new()
        along = einsum("lu,luv->lv", x, A).new()
        switch = Tensor(gb.dtypes.FP64, (L, n))
        switch(degree.S) << (x.sum(axis=0).new() - x) * omega
        new = teleport.ewise_add((along + switch) * d, binary.plus).new()
        err = (new - r).new().apply(unary.abs).new().sum().new().to_scalar().value
        r = new
        if err < tol:
            break
    return r, it


r, iterations = multiplex_pagerank(A)
versatility = r.sum(axis=0).new().to_dense(fill_value=0)
iterations, r.nvals, versatility.sum()

(102, 2034, np.float64(0.9999999999999998))

## Checking it

The same PageRank solved exactly with scipy, on the supra-graph whose vertices are the pairs: `r = d P^T r + (1 - d) / N`, with `P` the walk's transition matrix, built here from the edge list alone.

In [8]:
def supra_transition(airline, src, dst, L, n, omega=1.0):
    present = np.zeros((L, n), bool)
    present[airline, src] = present[airline, dst] = True
    rows = [airline * n + src, airline * n + dst]
    cols = [airline * n + dst, airline * n + src]
    for a in range(L):
        for b in range(L):
            if a != b:
                both = np.nonzero(present[a] & present[b])[0]
                rows.append(a * n + both)
                cols.append(b * n + both)
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    weights = np.where(rows % n == cols % n, omega, 1.0)  # a switch keeps the airport
    W = sp.csr_matrix((weights, (rows, cols)), shape=(L * n, L * n))
    pairs = np.flatnonzero(present)
    W = W[pairs][:, pairs]
    return sp.diags(1 / np.asarray(W.sum(axis=1)).ravel()) @ W, pairs


P, pairs = supra_transition(airline, src, dst, L, n)
N, d = len(pairs), 0.85
exact = spla.spsolve((sp.identity(N) - d * P.T).tocsc(), np.full(N, (1 - d) / N))
float(np.abs(exact - r.to_dense(fill_value=0).ravel()[pairs]).max())

9.86548082809735e-14

## The most versatile airports

Versatility next to plain counts: routes from the airport, and airlines serving it. Airports are named by their ICAO codes.

In [9]:
def table(versatility, top=12):
    print(f"{'rank':>4s}  {'airport':7s} {'versatility':>11s} {'routes':>6s} {'airlines':>8s}")
    for rank, i in enumerate(np.argsort(-versatility)[:top], 1):
        print(
            f"{rank:4d}  {airports[i]:7s} {versatility[i]:11.5f} {n_routes[i]:6.0f} {n_airlines[i]:8d}"
        )


table(versatility)

rank  airport versatility routes airlines
   1  LEBL        0.02463    137       31
   2  EHAM        0.02177    156       27
   3  LIRF        0.02074    119       29
   4  LGAV        0.02039    103       24
   5  EDDM        0.02014    139       26
   6  LEMD        0.02012    152       26
   7  EBBR        0.01927    104       27
   8  LOWW        0.01915    122       26
   9  EDDF        0.01840    127       25
  10  LIMC        0.01827     85       28
  11  LFPG        0.01691    118       24
  12  EDDL        0.01634    115       23


`omega` sets how freely the walker changes airline. Spearman's rank correlation of versatility with the number of routes and with the number of airlines, for a few values of `omega`:

In [10]:
used = n_routes > 0
for omega in [0.1, 1.0, 10.0]:
    v = multiplex_pagerank(A, omega=omega)[0].sum(axis=0).new().to_dense(fill_value=0)
    by_routes = spearmanr(v[used], n_routes[used])[0]
    by_airlines = spearmanr(v[used], n_airlines[used])[0]
    print(f"omega {omega:4}: with routes {by_routes:.3f}, with airlines {by_airlines:.3f}")

omega  0.1: with routes 0.975, with airlines 0.904
omega  1.0: with routes 0.941, with airlines 0.947


omega 10.0: with routes 0.871, with airlines 0.973


A small `omega` keeps the walker on one airline, so versatility follows the number of routes; a large one lets it change airline freely, so versatility follows the number of airlines serving the airport. At `omega = 1`, Milan Malpensa (LIMC) ranks tenth with 85 routes, because 28 airlines serve it.